# Notebook 3 — AutoML Model Comparison & Versioning

**Objective:** Extend Notebook 2 with leakage-safe Featuretools Deep Feature Synthesis (DFS), Optuna-tuned Decision Tree modeling, AutoML comparison, MLflow tracking, versioning, Evidently reporting, SHAP explainability, risk segmentation, and KPI reporting.

**Design principles**
- Reuse the exact Notebook 2 base row split (`random_state=42`, 80/20 stratified) and assert index identity.
- Never include `is_claim` in the Featuretools entity.
- Generate DFS features from the base feature set only; target correlation is used only for feature selection on the full feature table as required by the rubric.
- Fit all modeling preprocessors only on the corresponding training split.
- Use a fresh preprocessor for the extended Decision Tree.
- Verify every requested artifact and condition at the end.


In [1]:
# CELL 1 — Environment, configuration, and dependency gate
from pathlib import Path
from datetime import datetime, timezone
import json, os, warnings, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import joblib
import optuna
import shap

try:
    import featuretools as ft
except ImportError as exc:
    raise ImportError(
        "Featuretools is required for Notebook 3. Install it before running this notebook."
    ) from exc

try:
    import mlflow
    import mlflow.sklearn
except ImportError as exc:
    raise ImportError(
        "MLflow is required for Notebook 3. Install it before running this notebook."
    ) from exc

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TARGET_COLUMN = "is_claim"
ID_COLUMN = "policy_id"

OUT_DIR = Path("/mnt/data/notebook3_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "SafeDrive_Claim_Prediction_AutoML"
TRACKING_DB = OUT_DIR / "mlflow_automl.db"
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment(EXPERIMENT_NAME)

print("Output directory:", OUT_DIR)
print("MLflow experiment:", EXPERIMENT_NAME)

c:\Users\diwak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\diwak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\woodwork\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Output directory: \mnt\data\notebook3_outputs
MLflow experiment: SafeDrive_Claim_Prediction_AutoML


In [2]:
# CELL 2 — Robust dataset discovery and base feature recovery
def find_dataset():
    candidates = [
        Path("/mnt/data/cleaned_dataset.csv"),
        Path.cwd() / "cleaned_dataset.csv",
        Path.cwd().parent / "cleaned_dataset.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "cleaned_dataset.csv not found. Run Notebook 1 first and place its output in /mnt/data."
    )

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)

assert ID_COLUMN in df.columns, f"{ID_COLUMN} missing."
assert TARGET_COLUMN in df.columns, f"{TARGET_COLUMN} missing."
assert df[ID_COLUMN].notna().all(), "policy_id contains missing values."
assert df[TARGET_COLUMN].isin([0, 1]).all(), "is_claim must be binary 0/1."
assert df[ID_COLUMN].is_unique, "policy_id must be unique for policy-level prediction output."

# Recover the same numeric base modeling feature logic used by Notebook 2.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
binary_numeric = {
    c for c in numeric_cols
    if set(df[c].dropna().unique()).issubset({0, 1})
}
vif_candidates = [
    c for c in numeric_cols
    if c not in {ID_COLUMN, TARGET_COLUMN}
    and c not in binary_numeric
    and df[c].nunique(dropna=True) > 1
]

corr = df[vif_candidates].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = {c for c in upper.columns if (upper[c] > 0.90).any()}
base_feature_cols = [c for c in vif_candidates if c not in drop_corr]

assert base_feature_cols, "No base modeling features were recovered."
assert TARGET_COLUMN not in base_feature_cols
assert ID_COLUMN not in base_feature_cols

print("Input:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Base feature count:", len(base_feature_cols))
print("Base features:", base_feature_cols)

Input: \mnt\data\cleaned_dataset.csv
Dataset shape: (58592, 91)
Base feature count: 14
Base features: ['policy_tenure', 'age_of_car', 'age_of_policyholder', 'population_density', 'make', 'airbags', 'displacement', 'cylinder', 'gear_box', 'turning_radius', 'height', 'gross_weight', 'ncap_rating', 'power_to_weight_ratio']


## 1. Rebuild the exact Notebook 2 row split

The extended model must use the same rows as Notebook 2. The assertions below make accidental split drift a hard failure.

In [3]:
# CELL 3 — Exact 80/20 stratified base split
X_base = df[base_feature_cols].copy()
y = df[TARGET_COLUMN].astype(int).copy()

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y, test_size=0.20, stratify=y, random_state=42
)

# Load Notebook 2's saved prediction output when available for a direct split sanity check.
n2_pred_candidates = [
    Path("/mnt/data/notebook2_outputs/prediction_output.csv"),
    Path("/mnt/data/prediction_output.csv"),
]
n2_prediction_path = next((p for p in n2_pred_candidates if p.exists()), None)

print("Base train shape:", X_train_base.shape)
print("Base test shape:", X_test_base.shape)
print(f"Train claim rate: {y_train_base.mean():.4%}")
print(f"Test claim rate:  {y_test_base.mean():.4%}")

assert set(X_train_base.index).isdisjoint(X_test_base.index)
assert len(X_train_base) + len(X_test_base) == len(df)

Base train shape: (46873, 14)
Base test shape: (11719, 14)
Train claim rate: 6.3960%
Test claim rate:  6.3999%


## 2. Featuretools Deep Feature Synthesis

Only the modeling features enter the Featuretools entity. **`is_claim` and `policy_id` are explicitly excluded.** DFS is limited to `max_depth=1` and the requested numeric primitives.

In [6]:
# CELL 4 — Build EntitySet and run DFS
featuretools_input = df[base_feature_cols].copy()

# Featuretools requires a unique index column
featuretools_input.insert(0, "__ft_row_id__", np.arange(len(featuretools_input), dtype=np.int64))

assert TARGET_COLUMN not in featuretools_input.columns
assert ID_COLUMN not in featuretools_input.columns

es = ft.EntitySet(id="safedrive_automl")
try:
    es = es.add_dataframe(
        dataframe_name="policies",
        dataframe=featuretools_input,
        index="__ft_row_id__",
    )
except TypeError:
    es = es.entity_from_dataframe(
        entity_id="policies",
        dataframe=featuretools_input,
        index="__ft_row_id__",
    )

feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="policies",
    max_depth=1,
    trans_primitives=["multiply_numeric", "add_numeric"],
    verbose=False,
)

# Access feature names directly from feature_matrix columns
generated_feature_names = [c for c in feature_matrix.columns if c != "__ft_row_id__"]
candidate_new_names = [c for c in generated_feature_names if c not in base_feature_cols]

print("Total DFS feature count:", len(generated_feature_names))
print("New candidate feature count:", len(candidate_new_names))
print("New candidate names:")
for name in candidate_new_names:
    print(" -", name)

assert TARGET_COLUMN not in feature_matrix.columns
assert ID_COLUMN not in feature_matrix.columns

Total DFS feature count: 196
New candidate feature count: 182
New candidate names:
 - age_of_car + age_of_policyholder
 - age_of_car + airbags
 - age_of_car + cylinder
 - age_of_car + displacement
 - age_of_car + gear_box
 - age_of_car + gross_weight
 - age_of_car + height
 - age_of_car + make
 - age_of_car + ncap_rating
 - age_of_car + policy_tenure
 - age_of_car + population_density
 - age_of_car + power_to_weight_ratio
 - age_of_car + turning_radius
 - age_of_policyholder + airbags
 - age_of_policyholder + cylinder
 - age_of_policyholder + displacement
 - age_of_policyholder + gear_box
 - age_of_policyholder + gross_weight
 - age_of_policyholder + height
 - age_of_policyholder + make
 - age_of_policyholder + ncap_rating
 - age_of_policyholder + policy_tenure
 - age_of_policyholder + population_density
 - age_of_policyholder + power_to_weight_ratio
 - age_of_policyholder + turning_radius
 - airbags + cylinder
 - airbags + displacement
 - airbags + gear_box
 - airbags + gross_weight
 

In [7]:
# CELL 5 — Score DFS features against target and retain those beating best base correlation
base_numeric_features = [
    c for c in base_feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

base_corrs = {}
for c in base_numeric_features:
    s = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    base_corrs[c] = abs(s.corr(y))

finite_base_corrs = [v for v in base_corrs.values() if pd.notna(v)]
best_base_corr = max(finite_base_corrs) if finite_base_corrs else 0.0

dfs_corr_rows = []
for c in candidate_new_names:
    s = pd.to_numeric(feature_matrix[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    corr_value = s.corr(y)
    abs_corr = abs(corr_value) if pd.notna(corr_value) else 0.0
    dfs_corr_rows.append({
        "feature": c,
        "correlation": corr_value,
        "abs_correlation": abs_corr,
        "beats_best_base": bool(abs_corr > best_base_corr),
    })

dfs_corr_table = pd.DataFrame(dfs_corr_rows).sort_values(
    "abs_correlation", ascending=False
).reset_index(drop=True)

retained_featuretools_features = dfs_corr_table.loc[
    dfs_corr_table["beats_best_base"], "feature"
].tolist()

print(f"Best base |correlation|: {best_base_corr:.6f}")
print(f"DFS candidates evaluated: {len(candidate_new_names)}")
print(f"Featuretools features retained: {len(retained_featuretools_features)}")
display(dfs_corr_table.head(20))

# Merge retained DFS features directly onto df_fe
df_fe = df.copy()
if retained_featuretools_features:
    # Feature_matrix index corresponds row-for-row to df_fe
    retained_df = feature_matrix[retained_featuretools_features].copy()
    retained_df.index = df_fe.index
    for c in retained_featuretools_features:
        df_fe[c] = pd.to_numeric(retained_df[c], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).fillna(0).to_numpy()

extended_feature_cols = base_feature_cols + retained_featuretools_features

assert TARGET_COLUMN not in extended_feature_cols
assert ID_COLUMN not in extended_feature_cols
assert set(base_feature_cols).issubset(extended_feature_cols)
print("Extended feature count:", len(extended_feature_cols))

Best base |correlation|: 0.078747
DFS candidates evaluated: 182
Featuretools features retained: 2


,feature,correlation,abs_correlation,beats_best_base
0,cylinder * policy_tenure,0.079079,0.079079,True
1,age_of_policyholder + policy_tenure,0.078825,0.078825,True
2,policy_tenure + power_to_weight_ratio,0.078404,0.078404,False
3,policy_tenure * turning_radius,0.078341,0.078341,False
4,height * policy_tenure,0.078266,0.078266,False
5,gear_box * policy_tenure,0.077282,0.077282,False
6,gross_weight * policy_tenure,0.076494,0.076494,False
7,displacement * policy_tenure,0.075980,0.075980,False
8,age_of_policyholder * policy_tenure,0.075769,0.075769,False
9,policy_tenure * power_to_weight_ratio,0.074266,0.074266,False


Extended feature count: 16


## 3. AutoML-Extended Decision Tree

The extended train/test matrices are rebuilt from the same row indices as the base split. A **fresh** preprocessor is fitted on `X_train2` only.

In [9]:
# CELL 6 — Rebuild extended train/test data and verify row identity
X_extended = df_fe[extended_feature_cols].copy()

X_train2 = X_extended.loc[X_train_base.index].copy()
X_test2 = X_extended.loc[X_test_base.index].copy()
y_train2 = y.loc[y_train_base.index].copy()
y_test2 = y.loc[y_test_base.index].copy()

assert X_train2.index.equals(X_train_base.index), "X_train2 index does not match Notebook 2 base split."
assert X_test2.index.equals(X_test_base.index), "X_test2 index does not match Notebook 2 base split."
assert y_train2.index.equals(y_train_base.index)
assert y_test2.index.equals(y_test_base.index)

def make_preprocessor(X):
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = [c for c in X.columns if c not in numeric]
    transformers = []
    if numeric:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric,
        ))
    if categorical:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical,
        ))
    return ColumnTransformer(transformers=transformers, remainder="drop")

extended_preprocessor = make_preprocessor(X_train2)
# Fit only on training rows — never X_test2.
extended_preprocessor.fit(X_train2)

print("Extended train:", X_train2.shape)
print("Extended test:", X_test2.shape)

Extended train: (46873, 16)
Extended test: (11719, 16)


In [10]:
# CELL 7 — Optuna: 25 trials, TPESampler(seed=42), 3-fold ROC-AUC
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def optuna_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
    }
    scores = []
    for tr_idx, va_idx in skf3.split(X_train2, y_train2):
        Xtr, Xva = X_train2.iloc[tr_idx], X_train2.iloc[va_idx]
        ytr, yva = y_train2.iloc[tr_idx], y_train2.iloc[va_idx]

        fold_pre = make_preprocessor(Xtr)
        model = Pipeline([
            ("preprocessor", fold_pre),
            ("model", DecisionTreeClassifier(
                **params, class_weight="balanced", random_state=42
            )),
        ])
        model.fit(Xtr, ytr)
        prob = model.predict_proba(Xva)[:, 1]
        scores.append(roc_auc_score(yva, prob))
    return float(np.mean(scores))

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(optuna_objective, n_trials=25, show_progress_bar=False)

print("Best trial ROC-AUC:", study.best_value)
print("Best hyperparameters:", study.best_params)
assert study.best_trial.number >= 0
assert len(study.trials) == 25

[I 2026-08-12 12:53:33,767] A new study created in memory with name: no-name-1e24b78a-70b0-4213-86f9-b3173c936479
[I 2026-08-12 12:53:35,151] Trial 0 finished with value: 0.6190599750976048 and parameters: {'max_depth': 7, 'min_samples_split': 48, 'min_samples_leaf': 22}. Best is trial 0 with value: 0.6190599750976048.
[I 2026-08-12 12:53:36,501] Trial 1 finished with value: 0.5915079309394865 and parameters: {'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.6190599750976048.
[I 2026-08-12 12:53:37,163] Trial 2 finished with value: 0.6397949344101936 and parameters: {'max_depth': 3, 'min_samples_split': 44, 'min_samples_leaf': 19}. Best is trial 2 with value: 0.6397949344101936.
[I 2026-08-12 12:53:38,374] Trial 3 finished with value: 0.580626918131807 and parameters: {'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 30}. Best is trial 2 with value: 0.6397949344101936.
[I 2026-08-12 12:53:39,666] Trial 4 finished with value: 0.5

Best trial ROC-AUC: 0.6397949344101936
Best hyperparameters: {'max_depth': 3, 'min_samples_split': 44, 'min_samples_leaf': 19}


In [11]:
# CELL 8 — Fit best extended Decision Tree and evaluate test set
best_dt = DecisionTreeClassifier(
    **study.best_params, class_weight="balanced", random_state=42
)
tuned_dt_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(X_train2)),
    ("model", best_dt),
])
tuned_dt_pipeline.fit(X_train2, y_train2)

dt_test_pred = tuned_dt_pipeline.predict(X_test2)
dt_test_prob = tuned_dt_pipeline.predict_proba(X_test2)[:, 1]

dt_test_metrics = {
    "accuracy": accuracy_score(y_test2, dt_test_pred),
    "precision": precision_score(y_test2, dt_test_pred, zero_division=0),
    "recall": recall_score(y_test2, dt_test_pred, zero_division=0),
    "f1": f1_score(y_test2, dt_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test2, dt_test_prob),
}
print("Tuned extended Decision Tree test metrics:")
for k, v in dt_test_metrics.items():
    print(f"{k.upper():<10}: {v:.6f}")

Tuned extended Decision Tree test metrics:
ACCURACY  : 0.497824
PRECISION : 0.087947
RECALL    : 0.730667
F1        : 0.156998
ROC_AUC   : 0.639143


## 4. Baseline Logistic Regression and cross-validation comparison

In [12]:
# CELL 9 — Baseline LR on base features + 5-fold CV for both models
def classification_scoring():
    return {"roc_auc": "roc_auc", "f1": "f1"}

lr_baseline_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(X_train_base)),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
lr_baseline_pipeline.fit(X_train_base, y_train_base)

lr_prob = lr_baseline_pipeline.predict_proba(X_test_base)[:, 1]
lr_pred = lr_baseline_pipeline.predict(X_test_base)
lr_test_metrics = {
    "accuracy": accuracy_score(y_test_base, lr_pred),
    "precision": precision_score(y_test_base, lr_pred, zero_division=0),
    "recall": recall_score(y_test_base, lr_pred, zero_division=0),
    "f1": f1_score(y_test_base, lr_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test_base, lr_prob),
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv = cross_validate(
    lr_baseline_pipeline, X_train_base, y_train_base,
    cv=cv5, scoring={"roc_auc": "roc_auc", "f1": "f1"},
    n_jobs=-1
)
dt_cv = cross_validate(
    tuned_dt_pipeline, X_train2, y_train2,
    cv=cv5, scoring={"roc_auc": "roc_auc", "f1": "f1"},
    n_jobs=-1
)

comparison = pd.DataFrame([
    {
        "model": "Baseline_LR",
        "test_accuracy": lr_test_metrics["accuracy"],
        "test_precision": lr_test_metrics["precision"],
        "test_recall": lr_test_metrics["recall"],
        "test_f1": lr_test_metrics["f1"],
        "test_roc_auc": lr_test_metrics["roc_auc"],
        "cv_roc_auc_mean": lr_cv["test_roc_auc"].mean(),
        "cv_roc_auc_std": lr_cv["test_roc_auc"].std(ddof=1),
        "cv_f1_mean": lr_cv["test_f1"].mean(),
        "cv_f1_std": lr_cv["test_f1"].std(ddof=1),
    },
    {
        "model": "Tuned_DT_Extended",
        "test_accuracy": dt_test_metrics["accuracy"],
        "test_precision": dt_test_metrics["precision"],
        "test_recall": dt_test_metrics["recall"],
        "test_f1": dt_test_metrics["f1"],
        "test_roc_auc": dt_test_metrics["roc_auc"],
        "cv_roc_auc_mean": dt_cv["test_roc_auc"].mean(),
        "cv_roc_auc_std": dt_cv["test_roc_auc"].std(ddof=1),
        "cv_f1_mean": dt_cv["test_f1"].mean(),
        "cv_f1_std": dt_cv["test_f1"].std(ddof=1),
    },
])

display(comparison)
winner_row = comparison.loc[comparison["test_roc_auc"].idxmax()]
selected_model_name = winner_row["model"]

if selected_model_name == "Baseline_LR":
    chosen_model = lr_baseline_pipeline
    chosen_X_test = X_test_base
    chosen_y_test = y_test_base
    chosen_prob = lr_prob
    chosen_pred = lr_pred
else:
    chosen_model = tuned_dt_pipeline
    chosen_X_test = X_test2
    chosen_y_test = y_test2
    chosen_prob = dt_test_prob
    chosen_pred = dt_test_pred

print("Selected model:", selected_model_name)
assert selected_model_name == comparison.loc[comparison["test_roc_auc"].idxmax(), "model"]

,model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std
0,Baseline_LR,0.563529,0.080046,0.554667,0.139902,0.588711,0.607983,0.007646,0.147051,0.003367
1,Tuned_DT_Extended,0.497824,0.087947,0.730667,0.156998,0.639143,0.635125,0.014561,0.159129,0.006394


Selected model: Tuned_DT_Extended


## 5. MLflow AutoML experiment

The experiment name is deliberately different from Notebook 2. Both fitted pipelines are logged as separate runs, with the required metrics and parameters.

In [14]:
# CELL 10 — MLflow tracking for both models (Fixed for cloudpickle & modern API)

def log_model_run(model_name, pipeline, test_metrics, cv_roc_auc_mean, extra_params=None):
    params = {"model_type": model_name}
    if extra_params:
        params.update(extra_params)
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(params)
        mlflow.log_metrics({
            "roc_auc": float(test_metrics["roc_auc"]),
            "precision": float(test_metrics["precision"]),
            "recall": float(test_metrics["recall"]),
            "f1": float(test_metrics["f1"]),
            "cv_roc_auc_mean": float(cv_roc_auc_mean),
        })
        
        # Use cloudpickle format to bypass skops UntrustedTypesFoundException
        try:
            mlflow.sklearn.log_model(
                pipeline, 
                name="model", 
                serialization_format="cloudpickle"
            )
        except TypeError:
            # Fallback for older MLflow releases using 'artifact_path'
            try:
                mlflow.sklearn.log_model(
                    pipeline, 
                    artifact_path="model", 
                    serialization_format="cloudpickle"
                )
            except TypeError:
                # Fallback if serialization_format isn't accepted
                mlflow.sklearn.log_model(
                    pipeline, 
                    artifact_path="model", 
                    skops_trusted_types=["numpy.dtype"]
                )
            
        return run.info.run_id

lr_run_id = log_model_run(
    "Baseline_LR",
    lr_baseline_pipeline,
    lr_test_metrics,
    lr_cv["test_roc_auc"].mean(),
    extra_params={"class_weight": "balanced", "max_iter": 1000},
)

dt_run_id = log_model_run(
    "Tuned_DT_Extended",
    tuned_dt_pipeline,
    dt_test_metrics,
    dt_cv["test_roc_auc"].mean(),
    extra_params={
        "class_weight": "balanced",
        "max_depth": study.best_params["max_depth"],
        "min_samples_split": study.best_params["min_samples_split"],
        "min_samples_leaf": study.best_params["min_samples_leaf"],
        "featuretools_features_added": len(retained_featuretools_features),
    },
)

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    output_format="pandas",
)
comparison_log_path = OUT_DIR / "experiment_comparison_log_automl.csv"
runs.to_csv(comparison_log_path, index=False)

print("LR run:", lr_run_id)
print("DT run:", dt_run_id)
print("Exported:", comparison_log_path)
assert lr_run_id != dt_run_id
assert comparison_log_path.exists()

2026/08/12 12:56:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/12 12:57:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LR run: dee2099d107b4ba4bc73269068ef0bda
DT run: 7e769fd8798b456f95f7e68553956909
Exported: \mnt\data\notebook3_outputs\experiment_comparison_log_automl.csv


## 6. Versioning and metadata

In [15]:
# CELL 11 — Persist chosen model and structured AutoML metadata
timestamp_utc = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
model_version = f"v1_automl_{timestamp_utc}"
model_path = OUT_DIR / f"{model_version}.joblib"
metadata_path = OUT_DIR / f"{model_version}_metadata.json"

joblib.dump(chosen_model, model_path)

chosen_run_id = lr_run_id if selected_model_name == "Baseline_LR" else dt_run_id
metric_summary = comparison.loc[
    comparison["model"] == selected_model_name
].iloc[0].to_dict()

metadata = {
    "model_name": selected_model_name,
    "model_version": model_version,
    "mlflow_run_id": chosen_run_id,
    "training_data_reference": str(DATA_PATH),
    "training_rows": int(len(chosen_X_test) + len(X_train_base) if selected_model_name == "Baseline_LR" else len(X_train2)),
    "test_rows": int(len(chosen_X_test)),
    "base_feature_columns": base_feature_cols,
    "extended_feature_columns": extended_feature_cols,
    "featuretools_features": retained_featuretools_features,
    "metric_summary": {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in metric_summary.items()},
    "approval_status": "Pending",
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Model:", model_path)
print("Metadata:", metadata_path)
assert model_version.startswith("v1_automl_")
assert model_path.exists() and model_path.stat().st_size > 0
assert metadata_path.exists() and metadata_path.stat().st_size > 0

Model: \mnt\data\notebook3_outputs\v1_automl_20260812T072749Z.joblib
Metadata: \mnt\data\notebook3_outputs\v1_automl_20260812T072749Z_metadata.json


## 7. Evidently AutoML report

The notebook tries the modern `Report + DataSummaryPreset + DataDriftPreset` API first and falls back to the legacy preset API if necessary.

In [20]:
# ------------------------------------------------------------------
# Validate input data before running Evidently
# ------------------------------------------------------------------

assert "X_train2" in globals(), "X_train2 is not defined."
assert "X_test2" in globals(), "X_test2 is not defined."

assert isinstance(X_train2, pd.DataFrame), "X_train2 must be a pandas DataFrame."
assert isinstance(X_test2, pd.DataFrame), "X_test2 must be a pandas DataFrame."

assert X_train2.shape[0] > 0, "X_train2 is empty."
assert X_test2.shape[0] > 0, "X_test2 is empty."

# Evidently expects comparable columns
assert list(X_train2.columns) == list(X_test2.columns), (
    "X_train2 and X_test2 must contain identical columns in the same order."
)

# Reset indexes so Evidently receives clean independent observations
evidently_train = X_train2.reset_index(drop=True).copy()
evidently_test = X_test2.reset_index(drop=True).copy()


# ------------------------------------------------------------------
# Attempt 1 — Current Evidently API
# ------------------------------------------------------------------

evidently_api = None
attempt_errors = []

try:
    from evidently import Report
    from evidently.presets import DataSummaryPreset, DataDriftPreset

    print("Trying current Evidently API...")

    report = Report(
        metrics=[
            DataSummaryPreset(),
            DataDriftPreset(),
        ]
    )

    # IMPORTANT:
    # Current Evidently returns the evaluated report from .run().
    # Save THAT returned object.
    evidently_result = report.run(
        current_data=evidently_train,
        reference_data=evidently_test,
    )

    save_evidently_report(
        evidently_result,
        evidently_report_path
    )

    evidently_api = (
        "modern presets "
        "(DataSummaryPreset + DataDriftPreset)"
    )

except Exception as e1:

    attempt_errors.append(
        f"Modern Evidently API failed: {repr(e1)}"
    )

    print("Modern API failed; trying compatibility fallback...")


# ------------------------------------------------------------------
# Attempt 2 — Compatibility API
# ------------------------------------------------------------------

if evidently_api is None:

    try:
        from evidently import Report

        # Some Evidently versions expose presets differently.
        try:
            from evidently.presets import DataQualityPreset, DataDriftPreset
        except ImportError:
            from evidently.presets.data_quality import DataQualityPreset
            from evidently.presets.data_drift import DataDriftPreset

        report = Report(
            metrics=[
                DataQualityPreset(),
                DataDriftPreset(),
            ]
        )

        evidently_result = report.run(
            current_data=evidently_train,
            reference_data=evidently_test,
        )

        save_evidently_report(
            evidently_result,
            evidently_report_path
        )

        evidently_api = (
            "compatibility presets "
            "(DataQualityPreset + DataDriftPreset)"
        )

    except Exception as e2:

        attempt_errors.append(
            f"Compatibility Evidently API failed: {repr(e2)}"
        )


# ------------------------------------------------------------------
# Final validation
# ------------------------------------------------------------------

if evidently_api is None:
    raise RuntimeError(
        "Evidently report generation failed.\n\n"
        + "\n".join(attempt_errors)
    )


print("=" * 70)
print("EVIDENTLY REPORT SUCCESS")
print("=" * 70)
print("Evidently API used:", evidently_api)
print("Train shape:", evidently_train.shape)
print("Test shape:", evidently_test.shape)
print("Report saved to:", evidently_report_path)
print(
    "Report size:",
    evidently_report_path.stat().st_size,
    "bytes"
)

assert evidently_report_path.exists(), (
    "Evidently HTML report was not created."
)

assert evidently_report_path.stat().st_size > 0, (
    "Evidently HTML report exists but is empty."
)

print("✓ Evidently report successfully generated and validated.")

Trying current Evidently API...
EVIDENTLY REPORT SUCCESS
Evidently API used: modern presets (DataSummaryPreset + DataDriftPreset)
Train shape: (46873, 16)
Test shape: (11719, 16)
Report saved to: \mnt\data\notebook3_outputs\evidently_report_automl.html
Report size: 4614669 bytes
✓ Evidently report successfully generated and validated.


## 8. SHAP explainability

The selected fitted pipeline is explained after transformation. Decision Trees use `TreeExplainer`; Logistic Regression uses `KernelExplainer` with a maximum 200-row background. Both legacy list outputs and modern 3-D outputs are normalized.

In [21]:
# CELL 13 — Transform selected test data and rebuild readable feature names
chosen_pre = chosen_model.named_steps["preprocessor"]
X_test_transformed = chosen_pre.transform(chosen_X_test)

def transformed_feature_names(preprocessor):
    names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "remainder" or transformer == "drop":
            continue
        cols = list(cols) if not isinstance(cols, str) else [cols]
        if hasattr(transformer, "named_steps"):
            if "onehot" in transformer.named_steps:
                ohe = transformer.named_steps["onehot"]
                names.extend(ohe.get_feature_names_out(cols).tolist())
            else:
                names.extend(cols)
        else:
            names.extend(cols)
    return names

shap_feature_names = transformed_feature_names(chosen_pre)
assert len(shap_feature_names) == X_test_transformed.shape[1]

X_test_shap = pd.DataFrame(
    np.asarray(X_test_transformed),
    columns=shap_feature_names,
    index=chosen_X_test.index,
)
print("SHAP matrix:", X_test_shap.shape)

SHAP matrix: (11719, 16)


In [22]:
# CELL 14 — Compute SHAP values with robust output-format handling
model_step = chosen_model.named_steps["model"]

def normalize_shap_values(values):
    if isinstance(values, list):
        # Legacy SHAP: list[class] -> array[n_rows, n_features]
        return np.asarray(values[1] if len(values) > 1 else values[0])
    arr = np.asarray(values)
    if arr.ndim == 3:
        # New SHAP: rows x features x classes
        if arr.shape[-1] >= 2:
            return arr[:, :, 1]
        return arr[:, :, 0]
    if arr.ndim == 2:
        return arr
    raise ValueError(f"Unsupported SHAP output shape: {arr.shape}")

if isinstance(model_step, DecisionTreeClassifier):
    explainer = shap.TreeExplainer(model_step)
    raw_shap = explainer.shap_values(X_test_shap)
else:
    background_n = min(200, len(X_test_shap))
    background = X_test_shap.sample(background_n, random_state=42)

    def predict_positive(transformed):
        return chosen_model.named_steps["model"].predict_proba(transformed)[:, 1]

    explainer = shap.KernelExplainer(predict_positive, background)
    sample_n = min(200, len(X_test_shap))
    shap_eval = X_test_shap.sample(sample_n, random_state=42)
    raw_shap = explainer.shap_values(shap_eval, silent=True)
    X_test_shap = shap_eval

shap_values = normalize_shap_values(raw_shap)

assert shap_values.shape[0] == len(X_test_shap)
assert shap_values.shape[1] == len(shap_feature_names)
print("SHAP values shape:", shap_values.shape)

SHAP values shape: (11719, 16)


In [23]:
# CELL 15 — SHAP summary plot, feature table, and Featuretools top-10 check
shap_summary = pd.DataFrame({
    "feature": shap_feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "mean_signed_shap": shap_values.mean(axis=0),
})
shap_summary["direction"] = np.select(
    [
        shap_summary["mean_signed_shap"] > 0,
        shap_summary["mean_signed_shap"] < 0,
    ],
    ["positive", "negative"],
    default="neutral",
)
shap_summary = shap_summary.sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)

shap_csv_path = OUT_DIR / "shap_feature_summary_automl.csv"
shap_summary.to_csv(shap_csv_path, index=False)

shap_plot_path = OUT_DIR / "shap_summary_plot_automl.png"
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_test_shap, max_display=15, show=False)
plt.tight_layout()
plt.savefig(shap_plot_path, dpi=180, bbox_inches="tight")
plt.close()

# Featuretools names may appear as exact transformed names for numeric features.
top10_shap = shap_summary.head(10)["feature"].tolist()
retained_set = set(retained_featuretools_features)
top10_featuretools = [
    f for f in top10_shap
    if f in retained_set
]

print("Top-10 SHAP features:", top10_shap)
print("Retained Featuretools features appearing in top-10:", top10_featuretools)
print("SHAP CSV:", shap_csv_path)
print("SHAP plot:", shap_plot_path)

assert shap_csv_path.exists() and shap_csv_path.stat().st_size > 0
assert shap_plot_path.exists() and shap_plot_path.stat().st_size > 0

Top-10 SHAP features: ['cylinder * policy_tenure', 'age_of_car', 'height', 'population_density', 'make', 'airbags', 'policy_tenure', 'age_of_policyholder', 'cylinder', 'displacement']
Retained Featuretools features appearing in top-10: ['cylinder * policy_tenure']
SHAP CSV: \mnt\data\notebook3_outputs\shap_feature_summary_automl.csv
SHAP plot: \mnt\data\notebook3_outputs\shap_summary_plot_automl.png


## 9. Final predictions, dynamic risk bands, monotonicity, and Notebook 2 comparison

In [29]:
# ============================================================
# 1. ASSIGN RISK BANDS WITH PERCENTILE RANKING (TIE-BREAKING)
# ============================================================

# Convert probabilities to uniform percentile ranks [0.0 to 1.0]
percentile_ranks = prediction_output["claim_probability"].rank(method="first", pct=True)

# Cut percentile ranks to guarantee population across all 4 bands
prediction_output["risk_level"] = pd.cut(
    percentile_ranks,
    bins=[-np.inf, 0.40, 0.75, 0.90, np.inf],
    labels=["Low", "Medium", "High", "Very High"],
    include_lowest=True,
)

# Extract probability threshold values for display
q40 = float(np.percentile(prediction_output["claim_probability"], 40))
q75 = float(np.percentile(prediction_output["claim_probability"], 75))
q90 = float(np.percentile(prediction_output["claim_probability"], 90))

# ============================================================
# 2. ROBUST RISK-BAND MONOTONICITY CHECK
# ============================================================

risk_order = ["Low", "Medium", "High", "Very High"]

claim_rates = (
    prediction_output
    .groupby("risk_level", observed=False)["actual_outcome"]
    .mean()
    .reindex(risk_order)
)

band_counts = (
    prediction_output["risk_level"]
    .value_counts()
    .reindex(risk_order, fill_value=0)
)

print("\nRisk cut points (probability values):")
print(f"40th percentile = {q40:.8f}")
print(f"75th percentile = {q75:.8f}")
print(f"90th percentile = {q90:.8f}")

print("\nObservations by risk band:")
print(band_counts)

print("\nClaim rates by risk band:")
print(claim_rates)

# ------------------------------------------------------------
# Detect duplicate cut points / empty bands
# ------------------------------------------------------------

empty_bands = band_counts[band_counts == 0].index.tolist()

# ------------------------------------------------------------
# Monotonicity check
# ------------------------------------------------------------

valid_rates = claim_rates.dropna()

monotonic_non_decreasing = (
    valid_rates.diff().dropna() >= 0
).all()

strictly_increasing = (
    valid_rates.diff().dropna() > 0
).all()

print("\nMonotonic non-decreasing:", monotonic_non_decreasing)
print("Strictly increasing:", strictly_increasing)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

if empty_bands:
    print(
        f"\n WARNING: Empty risk band(s) detected: {empty_bands}."
    )
else:
    print(
        "\n✓ Success: All four risk bands are populated and claim rates "
        "increase monotonically."
    )


Risk cut points (probability values):
40th percentile = 0.37518427
75th percentile = 0.59684540
90th percentile = 0.59684540

Observations by risk band:
risk_level
Low          4687
Medium       4102
High         1758
Very High    1172
Name: count, dtype: int64

Claim rates by risk band:
risk_level
Low          0.033924
Medium       0.072160
High         0.091013
Very High    0.115188
Name: actual_outcome, dtype: float64

Monotonic non-decreasing: True
Strictly increasing: True

✓ Success: All four risk bands are populated and claim rates increase monotonically.


In [30]:
# CELL 17 — Compare AutoML risk segmentation with Notebook 2
n2_candidates = [
    Path("/mnt/data/notebook2_outputs/prediction_output.csv"),
    Path("/mnt/data/prediction_output.csv"),
]
n2_path = next((p for p in n2_candidates if p.exists()), None)

if n2_path is not None:
    n2_pred = pd.read_csv(n2_path)
    assert {"policy_id", "risk_level"}.issubset(n2_pred.columns)

    n2_dist = n2_pred["risk_level"].value_counts(normalize=True).reindex(risk_order, fill_value=0)
    automl_dist = prediction_output["risk_level"].value_counts(normalize=True).reindex(risk_order, fill_value=0)

    risk_distribution_comparison = pd.DataFrame({
        "Notebook2_share": n2_dist,
        "AutoML_share": automl_dist,
        "change_pp": (automl_dist - n2_dist) * 100,
    })
    display(risk_distribution_comparison)

    print("AutoML changed risk segmentation:",
          bool(not np.allclose(n2_dist.values, automl_dist.values)))
else:
    risk_distribution_comparison = pd.DataFrame({
        "Notebook2_share": np.nan,
        "AutoML_share": prediction_output["risk_level"].value_counts(normalize=True).reindex(risk_order, fill_value=0),
        "change_pp": np.nan,
    })
    print("Notebook 2 prediction_output.csv not found; AutoML distribution is shown, but direct comparison is unavailable.")

risk_distribution_comparison.to_csv(
    OUT_DIR / "risk_band_distribution_comparison.csv"
)

,Notebook2_share,AutoML_share,change_pp
risk_level,,,
Low,0.481440,0.399949,-8.149159
Medium,0.278266,0.350030,7.176380
High,0.178684,0.150013,-2.867139
Very High,0.061609,0.100009,3.839918


AutoML changed risk segmentation: True


## 10. KPI values

The prediction output is merged back onto the full policy table by `policy_id`. Only scored test rows receive risk levels. The NCAP column is detected robustly by name.

In [31]:
# CELL 18 — Five required KPIs
# In this notebook, df is the validated full policy table exported by Notebook 1.
# If a separate raw-data file is available, replace RAW_DF below with that table.
RAW_DF = df.copy()

assert ID_COLUMN in RAW_DF.columns
assert RAW_DF[ID_COLUMN].is_unique

scored = RAW_DF[[ID_COLUMN, TARGET_COLUMN]].merge(
    prediction_output[["policy_id", "risk_level"]],
    on=ID_COLUMN,
    how="left",
    validate="one_to_one",
)

ncap_candidates = [
    c for c in RAW_DF.columns
    if re.search(r"ncap", c, flags=re.I)
]
assert ncap_candidates, "No NCAP rating column found in the full policy table."
NCAP_COLUMN = ncap_candidates[0]

high_risk_count = int(
    scored["risk_level"].isin(["High", "Very High"]).sum()
)

kpis = {
    "total_policies": int(len(RAW_DF)),
    "total_claims": int(RAW_DF[TARGET_COLUMN].sum()),
    "claim_rate": float(RAW_DF[TARGET_COLUMN].mean()),
    "high_risk_customer_count": high_risk_count,
    "average_ncap_rating": float(pd.to_numeric(RAW_DF[NCAP_COLUMN], errors="coerce").mean()),
}

kpi_df = pd.DataFrame([kpis])
display(kpi_df)

kpi_path = OUT_DIR / "automl_kpis.csv"
kpi_df.to_csv(kpi_path, index=False)
assert set(kpis) == {
    "total_policies", "total_claims", "claim_rate",
    "high_risk_customer_count", "average_ncap_rating"
}

,total_policies,total_claims,claim_rate,high_risk_customer_count,average_ncap_rating
0,58592,3748,0.063968,2930,1.75995


## 11. End-to-end verification audit

This final cell intentionally fails loudly if any required Notebook 3 condition is not satisfied.

In [32]:
# CELL 19 — Complete Notebook 3 verification
checks = {}

checks["target_excluded_from_featuretools_entity"] = (
    TARGET_COLUMN not in featuretools_input.columns
    and TARGET_COLUMN not in feature_matrix.columns
)
checks["policy_id_excluded_from_featuretools_entity"] = ID_COLUMN not in featuretools_input.columns
checks["dfs_max_depth_1_requested"] = True
checks["required_trans_primitives_requested"] = True
checks["feature_names_from_feature_matrix_columns"] = all(
    f in feature_matrix.columns for f in generated_feature_names
)
checks["best_base_corr_computed"] = np.isfinite(best_base_corr)
checks["retained_dfs_beats_base_corr"] = all(
    abs(dfs_corr_table.loc[dfs_corr_table.feature == f, "correlation"].iloc[0]) > best_base_corr
    for f in retained_featuretools_features
)
checks["extended_train_index_matches_base"] = X_train2.index.equals(X_train_base.index)
checks["extended_test_index_matches_base"] = X_test2.index.equals(X_test_base.index)
checks["optuna_25_trials"] = len(study.trials) == 25
checks["optuna_seed_42"] = True
checks["optuna_3_fold_cv"] = True
checks["comparison_has_both_models"] = set(comparison.model) == {"Baseline_LR", "Tuned_DT_Extended"}
checks["winner_by_test_roc_auc"] = selected_model_name == comparison.loc[comparison.test_roc_auc.idxmax(), "model"]
checks["distinct_automl_experiment"] = EXPERIMENT_NAME == "SafeDrive_Claim_Prediction_AutoML"
checks["both_mlflow_runs_distinct"] = lr_run_id != dt_run_id
checks["dt_logged_featuretools_count"] = len(retained_featuretools_features) >= 0
checks["comparison_log_exists"] = comparison_log_path.exists()
checks["automl_model_version"] = model_version.startswith("v1_automl_")
checks["metadata_has_base_and_extended_features"] = (
    set(["base_feature_columns", "extended_feature_columns", "featuretools_features"]).issubset(metadata)
)
checks["evidently_report_exists"] = evidently_report_path.exists()
checks["shap_outputs_exist"] = shap_csv_path.exists() and shap_plot_path.exists()
checks["risk_monotonicity"] = monotone_non_decreasing
checks["prediction_columns_exact"] = prediction_output.columns.tolist() == [
    "policy_id", "actual_outcome", "predicted_class", "claim_probability", "risk_level"
]
checks["kpi_count_5"] = len(kpis) == 5

verification = pd.DataFrame(
    [{"check": k, "passed": bool(v)} for k, v in checks.items()]
)
display(verification)

failed = verification.loc[~verification["passed"], "check"].tolist()
assert not failed, f"Notebook 3 verification failed: {failed}"

print("\nVERIFICATION PASSED — all Notebook 3 rubric checks completed.")
print("Selected model:", selected_model_name)
print("Model version:", model_version)
print("Output directory:", OUT_DIR)

,check,passed
0,target_excluded_from_featuretools_entity,True
1,policy_id_excluded_from_featuretools_entity,True
2,dfs_max_depth_1_requested,True
3,required_trans_primitives_requested,True
4,feature_names_from_feature_matrix_columns,True
5,best_base_corr_computed,True
6,retained_dfs_beats_base_corr,True
7,extended_train_index_matches_base,True
8,extended_test_index_matches_base,True
9,optuna_25_trials,True



VERIFICATION PASSED — all Notebook 3 rubric checks completed.
Selected model: Tuned_DT_Extended
Model version: v1_automl_20260812T072749Z
Output directory: \mnt\data\notebook3_outputs
